# Формирование полного названия документа

Этап:
**Markdown из `output` → извлечение атрибутов в плоский JSON → ранее определённый № п/п → правило из `rules2.xlsx` → полное название документа**

Тематика на этом этапе не используется.


## 1. Импорты

In [ ]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI


## 2. Настройки

In [ ]:
BASE_URL = "https://ai-gateway.raisa.go.rshbank.ru/v1"
MODEL = "Qwen/Qwen3.6-35B-test/sovetnik"
API_KEY = "ВАШ_API_KEY"

OUTPUT_FOLDER = Path("./output")
RULES2_FILE = Path("./rules2.xlsx")

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    timeout=300.0,
)

client.models.list()


## 3. Чтение rules2.xlsx

In [ ]:
rules2_df = pd.read_excel(RULES2_FILE)

print("Столбцы rules2.xlsx:")
for col in rules2_df.columns:
    print("-", repr(col))

rules2_df.head()


## 4. Названия нужных столбцов

In [ ]:
# При необходимости замените на точные названия из Excel

COL_N_PP = "№ п/п"

COL_TITLE_RULE = (
    "В СЭД в РКК Входящего документа должен формироваться заголовок:"
)


## 5. Подготовка rules2

In [ ]:
rules2_clean = rules2_df[
    [COL_N_PP, COL_TITLE_RULE]
].copy()

rules2_clean = rules2_clean.dropna(
    how="all",
    subset=[COL_N_PP, COL_TITLE_RULE]
)

rules2_clean[COL_N_PP] = (
    rules2_clean[COL_N_PP]
    .fillna("")
    .astype(str)
    .str.strip()
)

rules2_clean[COL_TITLE_RULE] = (
    rules2_clean[COL_TITLE_RULE]
    .fillna("")
    .astype(str)
    .str.strip()
)

rules2_clean = rules2_clean[
    (rules2_clean[COL_N_PP] != "")
    & (rules2_clean[COL_TITLE_RULE] != "")
].reset_index(drop=True)

rules2_clean["_n_pp_norm"] = (
    rules2_clean[COL_N_PP]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
)

print("Правил:", len(rules2_clean))
rules2_clean.head()


## 6. Markdown-файлы в output

In [ ]:
md_files = sorted(OUTPUT_FOLDER.glob("*.md"))

print(f"Найдено .md файлов: {len(md_files)}")

for i, path in enumerate(md_files, start=1):
    print(f"{i}. {path.name}")


## 7. Выбор Markdown-файла

In [ ]:
MD_FILE = "1_цель.md"

md_path = OUTPUT_FOLDER / MD_FILE

if not md_path.exists():
    raise FileNotFoundError(f"Не найден файл: {md_path}")

document_text = md_path.read_text(
    encoding="utf-8"
)

print("Файл:", md_path.name)
print("Символов:", len(document_text))
print()
print(document_text[:3000])


## 8. Промт извлечения атрибутов

Замените примерный список на свои атрибуты.
JSON должен оставаться плоским.


In [ ]:
ATTRIBUTES_PROMPT = """
Извлеки из текста документа следующие атрибуты:

- номер_исполнительного_производства
- дата_исполнительного_производства
- наименование_должника

Правила:
- извлекай значения только из текста документа;
- ничего не придумывай;
- не исправляй значения;
- если значение отсутствует, верни null;
- верни только плоский JSON;
- никаких вложенных объектов;
- никаких массивов;
- никаких комментариев;
- названия ключей должны строго соответствовать указанным атрибутам.

Формат:
{
  "номер_исполнительного_производства": null,
  "дата_исполнительного_производства": null,
  "наименование_должника": null
}
"""


## 9. Извлечение атрибутов

In [ ]:
def extract_attributes(document_text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": ATTRIBUTES_PROMPT,
            },
            {
                "role": "user",
                "content": document_text,
            },
        ],
        temperature=0,
        response_format={"type": "json_object"},
        extra_body={
            "top_k": 20,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
        },
    )

    raw = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(
            "Модель вернула невалидный JSON:\n\n" + raw
        )

    for key, value in result.items():
        if isinstance(value, (dict, list)):
            raise ValueError(
                f"Поле '{key}' содержит вложенную структуру"
            )

    return result


## 10. Проверка извлечённых атрибутов

In [ ]:
attributes = extract_attributes(document_text)

print(
    json.dumps(
        attributes,
        ensure_ascii=False,
        indent=2
    )
)


## 11. Ранее определённый № п/п

Подставьте номер, полученный на предыдущем этапе классификации.
Тематика не нужна.


In [ ]:
N_PP = "2.1.1."

print("№ п/п:", N_PP)


## 12. Получение правила формирования названия

In [ ]:
def get_title_rule(n_pp: str) -> str:
    n_pp_norm = str(n_pp).strip().rstrip(".")

    matches = rules2_clean[
        rules2_clean["_n_pp_norm"] == n_pp_norm
    ]

    if matches.empty:
        raise ValueError(
            f"Для № п/п '{n_pp}' не найдено правило в rules2.xlsx"
        )

    return str(
        matches.iloc[0][COL_TITLE_RULE]
    ).strip()


title_rule = get_title_rule(N_PP)

print(title_rule)


## 13. Промт формирования полного названия

In [ ]:
TITLE_PROMPT = """
Сформируй полное название документа строго по переданному правилу.

Тебе будут переданы:
1. Правило формирования названия.
2. Атрибуты документа в плоском JSON.

Требования:
- строго следуй правилу;
- используй только значения из JSON;
- ничего не придумывай;
- учитывай все условия из правила;
- правильно подставляй атрибуты;
- сохраняй постоянный текст шаблона;
- не добавляй пояснений;
- не пиши "Название документа:";
- не оборачивай результат в кавычки;
- не возвращай JSON;
- верни только итоговое полное название документа.
"""


## 14. Формирование полного названия

In [ ]:
def generate_title(
    title_rule: str,
    attributes: dict
) -> str:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": TITLE_PROMPT,
            },
            {
                "role": "user",
                "content": f"""ПРАВИЛО ФОРМИРОВАНИЯ НАЗВАНИЯ:

{title_rule}

АТРИБУТЫ ДОКУМЕНТА:

{json.dumps(
    attributes,
    ensure_ascii=False,
    indent=2
)}
""",
            },
        ],
        temperature=0,
        extra_body={
            "top_k": 20,
            "chat_template_kwargs": {
                "enable_thinking": True
            },
        },
    )

    return response.choices[0].message.content.strip()


## 15. Финальный запуск

В output ячейки выводится только полное название документа.


In [ ]:
attributes = extract_attributes(document_text)

title_rule = get_title_rule(N_PP)

full_title = generate_title(
    title_rule,
    attributes
)

print(full_title)
